# Recitation 7: NLP & Image Analysis
### Data Science for Entrepreneurship-Economy | Spring 2026

**Today's goal**: Turn unstructured data (text and images) into numbers — then use your existing toolkit on them.

**Part 1 — NLP (~45 min)**: Preprocess text → Bag-of-Words → text similarity → classification
**Part 2 — Images (~45 min)**: Pixels as arrays → convolution filters → edge detection, blur, sharpen

By the end you'll have coded every concept from Lecture 7 that was previously just diagrams.

**What's new**: `CountVectorizer`, NLTK, `scipy.signal.convolve2d`. New libraries, familiar logic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Image imports
from scipy.signal import convolve2d

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("cheffi_meals.csv")
print(f"Loaded {len(df)} meal listings across {df['cuisine'].nunique()} cuisines\n")
print(df["cuisine"].value_counts())

> **Cheffi** (שפי) is an Israeli home-chef marketplace where amateur and semi-professional cooks list homemade meals for neighborhood delivery. Each listing has a **text description** and a **food photo**.
>
> Lior, the Head of Product, has two problems:
> 1. **Text**: Auto-tag listings by cuisine (Israeli, Asian, Italian, Middle Eastern, Vegan) from descriptions alone
> 2. **Image**: Understand image processing so the team can score and enhance photo quality
>
> Lior's first goal: auto-tag every listing by cuisine using only the text description.

In [ ]:
# Show 5 sample descriptions, one per cuisine
for cuisine in df["cuisine"].unique():
    sample = df[df["cuisine"] == cuisine].iloc[0]
    print(f"[{cuisine:15s}] {sample['description'][:80]}...")
    print()

---
## 📝 Section 2: Text Preprocessing
*Raw text → clean tokens. The NLP assembly line you saw in Lecture 7 — now in Python.*

---

### 📖 Text Preprocessing Pipeline

In Lecture 7 you traced through the pipeline by hand: lowercase → remove punctuation → tokenize → remove stopwords → lemmatize. Now you code each step. The good news: scikit-learn's `CountVectorizer` can do most of this automatically — but first, let's do it manually so you understand what's happening under the hood.

In [ ]:
sample_text = df.iloc[0]["description"]
print(f"Original: {sample_text}\n")

# Step 1: Lowercase
text = sample_text.lower()
print(f"Lowercase: {text}\n")

# Step 2: Tokenize
tokens = word_tokenize(text)
print(f"Tokens ({len(tokens)}): {tokens}\n")

# Step 3: Remove stopwords and non-alphabetic tokens
stop_words = set(stopwords.words("english"))
tokens_clean = [t for t in tokens if t.isalpha() and t not in stop_words]
print(f"After stopwords ({len(tokens_clean)}): {tokens_clean}\n")

# Step 4: Lemmatize
lemmatizer = WordNetLemmatizer()
tokens_lemma = [lemmatizer.lemmatize(t) for t in tokens_clean]
print(f"Lemmatized ({len(tokens_lemma)}): {tokens_lemma}")

In [ ]:
def preprocess(text):
    """Full preprocessing pipeline for one text string."""
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join([lemmatizer.lemmatize(t) for t in tokens])

df["clean_text"] = df["description"].apply(preprocess)
print("Before:", df.iloc[0]["description"][:80])
print("After: ", df.iloc[0]["clean_text"][:80])
print(f"\nAvg words before: {df['description'].str.split().str.len().mean():.1f}")
print(f"Avg words after:  {df['clean_text'].str.split().str.len().mean():.1f}")

### 🎯 Exercise 1: Preprocess a New Description

- Take this description: `"The BEST homemade Pasta with Fresh Basil and Creamy Tomato Sauce! Absolutely delicious."`
- Apply the `preprocess()` function to it
- How many words before vs. after?
- Which words were removed? Were any important words lost?

In [ ]:
##### Your code goes here #####

new_desc = "The BEST homemade Pasta with Fresh Basil and Creamy Tomato Sauce! Absolutely delicious."

# Apply preprocessing

# Words before vs after?

# Which words were removed? Any important ones lost? (comment)

###### End of Your Code ######

<details>
<summary><b>✅ Solution</b></summary>

```python
new_desc = "The BEST homemade Pasta with Fresh Basil and Creamy Tomato Sauce! Absolutely delicious."

clean = preprocess(new_desc)
print(f"Before ({len(new_desc.split())} words): {new_desc}")
print(f"After  ({len(clean.split())} words): {clean}")

# Removed: "The", "with", "and" (stopwords), "!" and "." (punctuation)
# "BEST" → "best" (lowercased). Key content words preserved.
# ~25% reduction — all the informative words remain.
```
</details>

---
## 🔢 Section 3: Bag-of-Words
*Turn clean text into a numeric matrix — the bridge from words to models.*

---

In [ ]:
# CountVectorizer does tokenization + counting in one step
bow_vectorizer = CountVectorizer(max_features=200)
X_bow = bow_vectorizer.fit_transform(df["clean_text"])

print(f"Document-Term Matrix shape: {X_bow.shape}")
print(f"  → {X_bow.shape[0]} documents × {X_bow.shape[1]} vocabulary words")
print(f"  → Sparsity: {1 - X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]):.1%}")

In [ ]:
vocab = bow_vectorizer.get_feature_names_out()
print(f"Top 20 words in vocabulary:\n{list(vocab[:20])}")
print(f"\nLast 20 words:\n{list(vocab[-20:])}")

In [ ]:
# Convert first document to a dense array and show non-zero entries
doc0 = pd.Series(X_bow[0].toarray()[0], index=vocab)
nonzero = doc0[doc0 > 0].sort_values(ascending=False)
print(f"Listing 0 — non-zero word counts ({len(nonzero)} words):\n")
print(nonzero)

### 🎯 Exercise 2: Find Similar Meals

- Pick listing 0 (or any listing you find interesting)
- Compute cosine similarity between listing 0's BoW vector and ALL other listings
- Print the **top 5 most similar** meals with their names, cuisines, and similarity scores
- Are they the same cuisine? Does the similarity make sense?

In [ ]:
##### Your code goes here #####

# Compute cosine similarity between listing 0 and all others

# Top 5 most similar (skip self)

# Are they the same cuisine? (comment)

###### End of Your Code ######

<details>
<summary><b>✅ Solution</b></summary>

```python
# Cosine similarity between listing 0 and all others
sim_scores = cosine_similarity(X_bow[0], X_bow)[0]

# Get top 6 (including self), then skip self (highest similarity = 1.0)
top_idx = sim_scores.argsort()[::-1][1:6]

print(f"Listing 0: {df.iloc[0]['description'][:60]}... [{df.iloc[0]['cuisine']}]\n")
print("Top 5 most similar:")
for i in top_idx:
    print(f"  Sim={sim_scores[i]:.3f} | [{df.iloc[i]['cuisine']:15s}] {df.iloc[i]['description'][:55]}...")

# Most similar listings likely share the same cuisine and key vocabulary.
# Cross-cuisine matches (e.g., Israeli ↔ Middle Eastern) are possible due to shared words.
```
</details>

### 💡 Insight

Text similarity using BoW + cosine is exactly how simple search engines work. When Lior's users search for 'spicy noodles,' the platform can rank listings by cosine similarity to the query — no machine learning model needed, just vector math.

---
## 🏷️ Section 4: Text Classification
*BoW gives you features. Classification gives you labels. Connect the dots.*

---

### 📖 From Text to Classifier

The document-term matrix IS a feature matrix — same structure as Recitations 3–4. Rows = documents, columns = word features, values = word counts. We can feed it directly into any classifier from Lecture 3. Today: Logistic Regression on BoW features to predict cuisine type.

In [ ]:
y = df["cuisine"]
X_train, X_test, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Classes: {list(y.unique())}")

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
ConfusionMatrixDisplay(cm, display_labels=clf.classes_).plot(cmap="Blues", xticks_rotation=45)
plt.title("Cuisine Classification — BoW + Logistic Regression")
plt.tight_layout()
plt.show()

### 📖 Reading the Results

Look at the confusion matrix: where does the model make mistakes? Israeli and Middle Eastern likely get confused (shared vocabulary like hummus, pita). That makes business sense — these cuisines genuinely overlap in ingredients and descriptions.

In [ ]:
feature_names = bow_vectorizer.get_feature_names_out()
for i, cuisine in enumerate(clf.classes_):
    top_idx = clf.coef_[i].argsort()[-8:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f"{cuisine:18s}: {', '.join(top_words)}")

### 📖 Most Important Words

These are the words the model relies on most for each cuisine. The model learned vocabulary-based rules from word counts — exactly what you'd expect from Bag-of-Words features.

### 🎯 Exercise 3: Classify a New Listing

- Write a new meal description (1–2 sentences) for a cuisine of your choice — but DON'T make it too obvious
- **Preprocess** it with `preprocess()`, then transform using `bow_vectorizer.transform([clean_text])`
- Predict the cuisine using the trained classifier
- Does the model get it right? If not, why might it fail?

In [ ]:
##### Your code goes here #####

new_listing = "..."  # Write your own description

# Transform with BoW vectorizer

# Predict cuisine

# Did it get it right? Why or why not? (comment)

###### End of Your Code ######

<details>
<summary><b>✅ Solution</b></summary>

```python
new_listing = "Creamy lentil soup with a squeeze of lemon and warm pita on the side"

# Preprocess, then transform
clean_new = preprocess(new_listing)
X_new = bow_vectorizer.transform([clean_new])

predicted = clf.predict(X_new)[0]
proba = clf.predict_proba(X_new)[0]

print(f"Description: {new_listing}")
print(f"Predicted cuisine: {predicted}\n")
for cls, p in zip(clf.classes_, proba):
    print(f"  {cls:18s}: {p:.1%}")

# This description is tricky — lentil/pita overlap Israeli, Middle Eastern, and Vegan.
# The model might misclassify because the vocabulary doesn't clearly signal one cuisine.
```
</details>

### 💡 Insight

From raw text to classified labels in 4 steps: preprocess → BoW → train → predict. The document-term matrix is the bridge. Every NLP classification task — spam detection, sentiment analysis, topic tagging — follows this exact pattern.

---
## 🖼️ Part 2: Image Analysis
*A photo is just a grid of numbers. Filters extract patterns from that grid.*

---

### 📖 Images as Arrays

In Lecture 7 you learned that an image is a 2D array of pixel values (0=black, 255=white for grayscale). You traced convolutions by hand on an 8×8 grid. Now you'll create images as numpy arrays, visualize them, and apply real filters in code.

In [ ]:
# Create a 24x24 grayscale image: bright circle (plate) on dark background (table)
img = np.full((24, 24), 30, dtype=float)   # dark table (value 30)

# Draw a bright circular plate in the center
center_row, center_col, radius = 12, 12, 8
for row in range(24):
    for col in range(24):
        if (row - center_row)**2 + (col - center_col)**2 <= radius**2:
            img[row, col] = 230   # bright plate

print(f"Image shape: {img.shape}")
print(f"Min pixel: {img.min():.0f}, Max pixel: {img.max():.0f}")

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.title("Synthetic Food Photo: Plate on Table")
plt.colorbar(label="Pixel Value")
plt.show()

In [ ]:
# Show a 6x6 patch at the plate edge (where the interesting transition happens)
patch = img[6:12, 6:12]
print("6×6 patch at the plate edge:\n")
print(patch.astype(int))

plt.figure(figsize=(4, 4))
plt.imshow(patch, cmap="gray", vmin=0, vmax=255)
for row in range(6):
    for col in range(6):
        plt.text(col, row, f"{int(patch[row,col])}", ha="center", va="center",
                 color="red" if patch[row,col] < 128 else "blue", fontsize=8)
plt.title("Zoomed In: Each Cell Is a Pixel Value")
plt.show()

The transition from 30 (dark table) to 230 (bright plate) is the plate edge. This is what edge detection filters look for.

In [ ]:
# Add random noise to simulate a bad phone camera
np.random.seed(42)
noise = np.random.normal(0, 25, img.shape)
img_noisy = np.clip(img + noise, 0, 255)

# Apply heavy blur to simulate an out-of-focus photo
blur_kernel = np.ones((5, 5)) / 25
img_blurry = convolve2d(img, blur_kernel, mode="same", boundary="fill")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Clean Photo")
axes[1].imshow(img_noisy, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Noisy Photo (bad camera)")
axes[2].imshow(img_blurry, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Blurry Photo (out of focus)")
plt.suptitle("Same Plate — Different Quality Issues", fontsize=13)
plt.tight_layout()
plt.show()

### 🎯 Exercise 4: Create Your Own Image

- Create a 24×24 grayscale image with a **bright square** (value 220) on a dark background (value 40)
- The square should be centered, roughly 10×10 pixels
- Display it with `plt.imshow()`
- Print the pixel values of a row that crosses the square edge

In [ ]:
##### Your code goes here #####

# Create 24x24 dark image

# Draw a bright 10x10 square in the center

# Display with imshow

# Print one row crossing the edge

###### End of Your Code ######

<details>
<summary><b>✅ Solution</b></summary>

```python
img2 = np.full((24, 24), 40, dtype=float)
img2[7:17, 7:17] = 220   # bright 10x10 square in the center

plt.figure(figsize=(5, 5))
plt.imshow(img2, cmap="gray", vmin=0, vmax=255)
plt.title("Bright Square on Dark Background")
plt.colorbar(label="Pixel Value")
plt.show()

# Print row 12 (crosses the square)
print(f"Row 12 pixel values:\n{img2[12].astype(int)}")
# Shows: 40 40 ... 40 220 220 ... 220 40 ... 40 — sharp edge transition
```
</details>

---
## 🔍 Section 5: Convolution Filters
*Slide a small number grid across the image. Multiply, sum, repeat. Detect edges, blur, sharpen.*

---

### 📖 Convolution Recap

In Lecture 7 you traced convolution by hand: place a 3×3 filter on the image, multiply each filter value by the pixel beneath it, sum the 9 products → one output number. Slide and repeat. The output is a 'feature map' that shows where the filter's pattern was detected. Now: code it.

In [ ]:
# scipy's convolve2d does the slide-multiply-sum for us
# mode='same' keeps output the same size as input
# boundary='fill' pads edges with zeros

def apply_filter(image, kernel, title="Filtered"):
    """Apply a filter (e.g. 3x3) and display before/after."""
    filtered = convolve2d(image, kernel, mode="same", boundary="fill")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Original")
    axes[1].imshow(filtered, cmap="gray")
    axes[1].set_title(title)
    plt.tight_layout()
    plt.show()
    return filtered

In [ ]:
# Vertical edge detection filter (Sobel-like)
edge_v = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])
print("Vertical Edge Filter:\n", edge_v)

edges_v = apply_filter(img, edge_v, "Vertical Edges Detected")

The filter produces high values where brightness changes sharply from left to right — the plate's left and right edges light up.

In [ ]:
# Horizontal edge detection filter
edge_h = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
])
edges_h = apply_filter(img, edge_h, "Horizontal Edges Detected")

In [ ]:
# Combine vertical and horizontal edges
edge_magnitude = np.sqrt(edges_v**2 + edges_h**2)

plt.figure(figsize=(5, 5))
plt.imshow(edge_magnitude, cmap="hot")
plt.title("Combined Edge Detection — Full Plate Rim Visible")
plt.colorbar(label="Edge Strength")
plt.show()

The plate's circular rim appears as a bright ring — the filter detected ALL the edges, regardless of direction.

In [ ]:
# Averaging (blur) filter — each pixel becomes the average of its 3x3 neighborhood
blur = np.ones((3, 3)) / 9
print("Blur Filter:\n", blur.round(3))

blurred = apply_filter(img_noisy, blur, "Blurred (Denoised)")

Applied to the noisy image, the blur filter smooths out the random noise while keeping the overall plate shape. This is a basic denoising technique.

In [ ]:
# Sharpen filter — amplifies differences from neighbors
sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
])
print("Sharpen Filter:\n", sharpen)

sharpened = apply_filter(img, sharpen, "Sharpened (Enhanced Edges)")

### 🎯 Exercise 5: Filter the Noisy Image

- Apply the **edge detection** filter (vertical) to the **noisy** image (`img_noisy`)
- Apply the **blur** filter to the noisy image FIRST, then apply edge detection to the blurred result
- Display all three: (1) edges on noisy image, (2) blurred image, (3) edges on blurred image
- Which gives cleaner edges? Why does blurring first help?

In [ ]:
##### Your code goes here #####

# 1. Edge detection on noisy image directly

# 2. Blur the noisy image first

# 3. Edge detection on the blurred version

# Which gives cleaner edges? Why? (comment)

###### End of Your Code ######

<details>
<summary><b>✅ Solution</b></summary>

```python
# 1. Edge detection directly on noisy image
noisy_edges = convolve2d(img_noisy, edge_v, mode="same", boundary="fill")

# 2. Blur first, then edge detection
blurred_noisy = convolve2d(img_noisy, blur, mode="same", boundary="fill")
clean_edges = convolve2d(blurred_noisy, edge_v, mode="same", boundary="fill")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(noisy_edges, cmap="gray")
axes[0].set_title("Edges on Noisy (messy)")
axes[1].imshow(blurred_noisy, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Blurred First")
axes[2].imshow(clean_edges, cmap="gray")
axes[2].set_title("Edges on Blurred (clean)")
plt.tight_layout()
plt.show()

# Blurring first smooths the noise, so edge detection finds the real plate rim cleanly.
# This blur-then-detect pipeline is standard in image processing.
# CNNs (Lecture 8) learn to do this automatically.
```
</details>

### 💡 Insight

For Lior's photo quality scorer: run edge detection on each food photo and compute mean edge magnitude. **Blurry** photos have weak edges (low magnitude) — the plate rim fades out. **Noisy** photos have artificially high edge magnitude — random noise creates false edges everywhere. Both are signs of low quality, detected differently: blurry = too few edges, noisy = too many spurious edges.

---
## 📊 Section 6: Putting It All Together

---

### 🗣️ Discussion: Lior's Full Report

*Instructor-led — no code. Think about how you'd present today's work to a non-technical CEO.*

> Lior presents to the CEO. What are his talking points?
>
> **NLP side:**
> 1. Can we auto-tag listings by cuisine? How accurate? Which cuisines get confused — and why?
> 2. What are the most important words for identifying each cuisine? (Look at the coefficients we printed above.)
>
> **Image side:**
> 3. How can we detect low-quality photos using filters? (Think: what does edge magnitude tell us about blurry vs. noisy photos?)
> 4. Can we auto-enhance blurry photos? What filter would help — and when would it fail?
>
> **Big picture:**
> 5. How does today's work connect to the models from Lectures 3–6? (Hint: the document-term matrix is just another feature matrix; image filters produce feature maps that CNNs will learn automatically in Lecture 8.)

---

### 🎓 What You Just Did

**NLP:**
- Preprocessed raw text: lowercase → tokenize → stopwords → lemmatize
- Built a Bag-of-Words document-term matrix with scikit-learn
- Measured text similarity using cosine similarity on BoW vectors
- Classified meal descriptions by cuisine using Logistic Regression

**Image Analysis:**
- Created grayscale images as numpy arrays
- Applied convolution filters: edge detection, blur, sharpen
- Discovered that blur + edge detection = clean feature extraction
- Designed a photo quality metric from edge strength

### The Unstructured Data Toolkit

```python
# NLP
CountVectorizer()              # text → word count matrix (BoW)
cosine_similarity(X_bow)       # find similar documents

# Images
convolve2d(image, kernel)      # apply a filter
edge_v, edge_h                 # detect edges
blur = np.ones((3,3)) / 9     # smooth noise
sharpen                        # enhance details
```

### 📅 What's Next

- **Lecture 8**: Deep Learning — models that learn their OWN features (no hand-designed filters or word counts)
- **Final Project**: Announced in Lecture 8 — apply everything from Lectures 1–7 to a real business problem

### 💬 One Last Thing

"Every previous recitation gave you structured data in rows and columns. Today you processed raw text and raw pixels — the 80% of business data that was previously invisible to your models. The document-term matrix and the feature map are the bridges. With these, your entire Lectures 3–6 toolkit works on text and images too. That's not a new skill — it's an extension of everything you already know."